# Polars Exercises 07 — Concat, Pivot & Unpivot

Two everyday reshaping jobs. First, **stacking** files that arrive in pieces — monthly exports, one file per source system — into one table with `concat()`. Second, changing a table's **shape**: `pivot()` turns long data into a wide cross-tab (like an Excel PivotTable), and `unpivot()` does the reverse, turning wide columns back into tidy long rows.


**Data files:** `data/orders_h1.csv`, `data/orders_h2.csv`, `data/orders.csv`

---

### 1. Import Polars and the selectors module (`cs`). Read the two half-year files `orders_h1.csv` and `orders_h2.csv`, and also the full `orders.csv`.

In [1]:
import polars as pl
import polars.selectors as cs

h1 = pl.read_csv("data/orders_h1.csv")
h2 = pl.read_csv("data/orders_h2.csv")
orders = pl.read_csv("data/orders.csv")
h1.shape, h2.shape, orders.shape

((513, 12), (487, 12), (1000, 12))

### 2. **Stack** `h1` on top of `h2` into one table with `pl.concat`.

In [2]:
combined = pl.concat([h1, h2])
combined.shape

(1000, 12)

### 3. Confirm the concat was lossless: the combined row count should equal `h1.height + h2.height`, and equal the full `orders` table.

In [3]:
combined = pl.concat([h1, h2])
combined.height, h1.height + h2.height, orders.height

(1000, 1000, 1000)

### 4. Diagonal concat: imagine `h2` gained a new column `source` that `h1` does not have. Add it to `h2`, then concat with `how="diagonal"` so the missing values become null.

In [4]:
h2b = h2.with_columns(source=pl.lit("system_B"))
pl.concat([h1, h2b], how="diagonal").select("order_id", "source").head()

order_id,source
str,str
"""ORD-000003""",null
"""ORD-000005""",null
"""ORD-000006""",null
"""ORD-000007""",null
"""ORD-000010""",null


### 5. **Pivot** (cross-tab): total `revenue` per `category` (rows) broken down by `region` (columns). First add a `revenue` column.

In [5]:
o = orders.with_columns(revenue=pl.col("quantity") * pl.col("unit_price"))
o.pivot(on="region", index="category", values="revenue", aggregate_function="sum")

category,Asia Pacific,North America,Europe,Latin America,Middle East & Africa
str,f64,f64,f64,f64,f64
"""pens""",1202.05,2349.77,1396.29,796.09,74.92
"""shirts""",12805.99,22466.83,12958.47,4824.82,1163.43
"""books""",7885.65,19004.98,11231.39,5329.67,1956.63
"""posters""",5743.89,10684.68,8070.04,3513.78,429.61
"""stickers""",420.53,1008.35,740.99,285.1,120.48
"""notebooks""",4874.32,7909.54,4372.05,2316.13,1123.27
"""mugs""",2548.8,4052.06,1920.19,1089.47,625.8


### 6. Pivot the **count of orders** by `region` (rows) × `channel` (columns).

In [6]:
orders.pivot(on="channel", index="region", values="order_id",
             aggregate_function="len")

region,online,partner,store
str,u32,u32,u32
"""Asia Pacific""",144,21,59
"""North America""",224,31,138
"""Europe""",145,24,72
"""Latin America""",71,9,30
"""Middle East & Africa""",19,2,11


### 7. Pivot the **average `unit_price`** by `category` (rows) × `channel` (columns), and round the numbers to 2 decimals.

In [7]:
orders.pivot(on="channel", index="category", values="unit_price",
             aggregate_function="mean").with_columns(
    cs.numeric().round(2)
)

category,online,partner,store
str,f64,f64,f64
"""pens""",1.96,2.05,1.71
"""shirts""",19.21,17.94,20.64
"""books""",14.42,9.89,14.77
"""posters""",7.76,8.16,8.39
"""stickers""",1.44,1.48,1.44
"""notebooks""",6.82,5.43,6.96
"""mugs""",6.29,6.32,6.44


### 8. A pivot can leave **null** cells when a combination never occurred. Take the revenue pivot and fill any nulls with `0`.

In [8]:
o = orders.with_columns(revenue=pl.col("quantity") * pl.col("unit_price"))
piv = o.pivot(on="region", index="category", values="revenue",
              aggregate_function="sum")
piv.with_columns(cs.numeric().fill_null(0))

category,Asia Pacific,North America,Europe,Latin America,Middle East & Africa
str,f64,f64,f64,f64,f64
"""pens""",1202.05,2349.77,1396.29,796.09,74.92
"""shirts""",12805.99,22466.83,12958.47,4824.82,1163.43
"""books""",7885.65,19004.98,11231.39,5329.67,1956.63
"""posters""",5743.89,10684.68,8070.04,3513.78,429.61
"""stickers""",420.53,1008.35,740.99,285.1,120.48
"""notebooks""",4874.32,7909.54,4372.05,2316.13,1123.27
"""mugs""",2548.8,4052.06,1920.19,1089.47,625.8


### 9. Add a **row total** to the revenue pivot: a `total` column summing all the region columns (use `pl.sum_horizontal` over the numeric columns).

In [9]:
o = orders.with_columns(revenue=pl.col("quantity") * pl.col("unit_price"))
piv = o.pivot(on="region", index="category", values="revenue",
              aggregate_function="sum").with_columns(cs.numeric().fill_null(0))
piv.with_columns(total=pl.sum_horizontal(cs.numeric()))

category,Asia Pacific,North America,Europe,Latin America,Middle East & Africa,total
str,f64,f64,f64,f64,f64,f64
"""pens""",1202.05,2349.77,1396.29,796.09,74.92,5819.12
"""shirts""",12805.99,22466.83,12958.47,4824.82,1163.43,54219.54
"""books""",7885.65,19004.98,11231.39,5329.67,1956.63,45408.32
"""posters""",5743.89,10684.68,8070.04,3513.78,429.61,28442.0
"""stickers""",420.53,1008.35,740.99,285.1,120.48,2575.45
"""notebooks""",4874.32,7909.54,4372.05,2316.13,1123.27,20595.31
"""mugs""",2548.8,4052.06,1920.19,1089.47,625.8,10236.32


### 10. **Unpivot** the revenue pivot back into tidy long form: one row per `category` + `region` with its `revenue`.

In [10]:
o = orders.with_columns(revenue=pl.col("quantity") * pl.col("unit_price"))
piv = o.pivot(on="region", index="category", values="revenue",
              aggregate_function="sum").with_columns(cs.numeric().fill_null(0))
piv.unpivot(index="category", variable_name="region", value_name="revenue")\
   .sort("category", "region")

category,region,revenue
str,str,f64
"""books""","""Asia Pacific""",7885.65
"""books""","""Europe""",11231.39
"""books""","""Latin America""",5329.67
"""books""","""Middle East & Africa""",1956.63
"""books""","""North America""",19004.98
…,…,…
"""stickers""","""Asia Pacific""",420.53
"""stickers""","""Europe""",740.99
"""stickers""","""Latin America""",285.1


### 11. **Unpivot** straight from `orders`: turn the `quantity` and `unit_price` columns into two rows per order, keeping `order_id` as the id.

In [11]:
orders.unpivot(
    on=["quantity", "unit_price"],
    index="order_id",
    variable_name="metric",
    value_name="value",
).head()

order_id,metric,value
str,str,f64
"""ORD-000001""","""quantity""",15.0
"""ORD-000002""","""quantity""",5.0
"""ORD-000003""","""quantity""",16.0
"""ORD-000004""","""quantity""",37.0
"""ORD-000005""","""quantity""",11.0


### 12. **Horizontal concat:** glue two same-height frames side by side — `orders.select("order_id")` and a one-column revenue frame (`how="horizontal"`).

In [12]:
left = orders.select("order_id")
right = orders.select(
    revenue=(pl.col("quantity") * pl.col("unit_price"))
)
pl.concat([left, right], how="horizontal").head()

order_id,revenue
str,f64
"""ORD-000001""",40.5
"""ORD-000002""",89.4
"""ORD-000003""",126.24
"""ORD-000004""",631.22
"""ORD-000005""",79.86


### 13. Build a clean **monthly revenue report**: from the combined table, pivot revenue by `region` (rows) × month (columns), fill nulls with 0. (derive month from `order_date`).

In [13]:
o = pl.concat([h1, h2]).with_columns(
    revenue=pl.col("quantity") * pl.col("unit_price"),
    month=pl.col("order_date").str.to_date("%Y-%m-%d").dt.month(),
)
o.pivot(on="month", index="region", values="revenue",
        aggregate_function="sum").with_columns(cs.numeric().fill_null(0))

region,2,5,6,1,3,4,10,12,8,11,7,9
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""Europe""",2663.85,3539.8,2321.11,5380.58,2915.38,2650.01,3142.67,3065.73,3938.94,2893.93,3690.21,4487.21
"""Asia Pacific""",2630.44,3626.6,2806.37,3431.54,2888.27,2944.8,3332.4,4009.36,3152.97,1384.27,3475.51,1798.7
"""North America""",7435.87,3907.95,6906.66,5366.18,4191.82,7753.19,5452.97,7723.8,6580.95,2410.89,5900.95,3844.98
"""Latin America""",1622.31,325.96,1173.12,1092.87,969.03,3966.52,1737.78,2680.25,580.37,1692.87,1437.18,876.8
"""Middle East & Africa""",570.74,252.08,11.07,415.44,136.5,262.73,53.68,1020.17,304.99,505.74,1113.89,847.11
